# JupyterLite（xeus-r）で学ぶ R 統計検定 入門チュートリアル

このノートブックは、ブラウザだけで動く JupyterLite の R カーネル（xeus-r）で、
**統計的仮説検定** の基本を一から学ぶチュートリアルです。
追加インストールは不要で、すべて R に付属する `stats` パッケージの関数だけを使います。

## 対象者
- [R 文法入門](r_beginner_tutorial.ipynb) を終えた方
- t 検定・カイ二乗検定・分散分析を「R でどう書くか」から学びたい方

## このチュートリアルで学ぶこと
1. 記述統計と分布の確認
2. t 検定（1 標本・2 標本・対応あり）
3. 分散の検定（F 検定）
4. 比率の検定
5. カイ二乗検定（適合度・独立性）
6. 一元配置分散分析（ANOVA）
7. ノンパラメトリック検定
8. 相関の検定
9. p 値と効果量の考え方
10. まとめと総合演習

## JupyterLite で使ううえでの注意
- **セルの最後の式だけ** が自動表示されます。途中の結果は `print()` や `cat()` で明示的に出力します
- グラフの中の文字は **英語のみ**（日本語フォントが無く豆腐になるため）。説明は Markdown 側で日本語で行います
- `install.packages()` は使えませんが、このノートは付属パッケージだけで完結します

---
## 0. 環境の準備

タイムゾーンとグラフの表示サイズを設定します。最初に必ず実行してください。

In [ ]:
# タイムゾーン設定（未設定だと Sys.time() などが警告を出す）
Sys.setenv(TZ = "Asia/Tokyo")

# グラフの表示サイズ（JupyterLite で見やすい設定）
options(repr.plot.width = 7, repr.plot.height = 4.5, repr.plot.res = 100, jupyter.plot_scale = 1)

cat("R のバージョン:", R.version.string, "\n")
cat("実行環境      :", Sys.info()[["sysname"]], "\n")   # JupyterLite では "Emscripten"

---
## 1. 記述統計と分布の確認

検定の前に、まずデータの**平均・ばらつき・分布の形**を確認するのが鉄則です。
ここでは 2 つのクラスの試験点数を乱数で作り、教材データとして使います
（`set.seed()` で乱数を固定しているので、誰が実行しても同じ結果になります）。

In [ ]:
set.seed(1)
scores_a <- round(rnorm(40, mean = 65, sd = 10))   # クラス A（40 人）の点数
scores_b <- round(rnorm(45, mean = 70, sd = 12))   # クラス B（45 人）の点数

cat("クラス A の先頭 10 人:", head(scores_a, 10), "\n")
cat("A: 平均", round(mean(scores_a), 2), "/ 中央値", median(scores_a),
    "/ 標準偏差", round(sd(scores_a), 2), "\n")
cat("B: 平均", round(mean(scores_b), 2), "/ 中央値", median(scores_b),
    "/ 標準偏差", round(sd(scores_b), 2), "\n")
summary(scores_a)

In [ ]:
# ヒストグラムと箱ひげ図で分布を確認（グラフ内の文字は英語で）
par(mfrow = c(1, 2))
hist(scores_a, main = "Class A scores", xlab = "Score", col = "skyblue")
boxplot(list(A = scores_a, B = scores_b), main = "Scores by class",
        ylab = "Score", col = c("skyblue", "salmon"))
par(mfrow = c(1, 1))

### 正規性の確認

t 検定などの多くの検定は「データが正規分布に近い」ことを仮定します。
- **Q-Q プロット**: 点が直線に乗っていれば正規分布に近い
- **シャピロ・ウィルク検定** `shapiro.test()`: 帰無仮説「正規分布に従う」。p 値が大きければ正規性を否定しない

In [ ]:
qqnorm(scores_a, main = "Normal Q-Q plot (Class A)")
qqline(scores_a, col = "red", lwd = 2)
shapiro.test(scores_a)

### 練習問題 1

`set.seed(100); waiting <- round(rexp(50, rate = 1/10), 1)` で「待ち時間（分）」のデータを作り、
1. 平均・中央値・標準偏差を表示してください
2. ヒストグラムを描いてください（タイトルは英語で）
3. `shapiro.test()` で正規性を検定し、結果を確認してください（指数分布なので正規性は棄却されるはずです）

In [ ]:
# 練習問題 1 の解答欄：ここにコードを書いてください

<details>
<summary><strong>練習問題 1 の解答例を見る</strong></summary>

```r
set.seed(100)
waiting <- round(rexp(50, rate = 1/10), 1)

cat("平均:", mean(waiting), "/ 中央値:", median(waiting), "/ 標準偏差:", round(sd(waiting), 2), "\n")
hist(waiting, main = "Waiting time", xlab = "Minutes", col = "lightgreen")
shapiro.test(waiting)   # p < 0.05 → 正規分布とは言えない
```

</details>

---
## 2. t 検定

**平均値に関する検定** の代表が t 検定です。`t.test()` 1 つで 3 種類の検定ができます。

| 種類 | 使う場面 | 書き方 |
|---|---|---|
| 1 標本 | 平均が特定の値と違うか | `t.test(x, mu = 60)` |
| 2 標本（独立） | 2 グループの平均が違うか | `t.test(x, y)` |
| 対応あり | 同じ対象の前後比較 | `t.test(after, before, paired = TRUE)` |

### 2.1 1 標本の t 検定

「クラス A の平均点は 60 点と言えるか？」を検定します。
- 帰無仮説 H0: 母平均 = 60
- 対立仮説 H1: 母平均 ≠ 60（両側検定）

In [ ]:
t.test(scores_a, mu = 60)

### 出力の読み方

- `t = ...`: 検定統計量（t 値）
- `df = ...`: 自由度（標本サイズ − 1）
- `p-value`: **帰無仮説が正しいとしたとき、これほど極端なデータが得られる確率**。
  慣習的に 0.05 より小さければ「有意差あり」として帰無仮説を棄却します
- `95 percent confidence interval`: 母平均の 95% 信頼区間。この区間に 60 が入っていなければ p < 0.05 と対応します
- `mean of x`: 標本平均

### 2.2 2 標本の t 検定（独立な 2 群）

クラス A とクラス B の平均点に差があるかを検定します。
R の既定は **Welch の t 検定**（2 群の分散が等しいことを仮定しない、頑健な方法）です。

In [ ]:
t.test(scores_a, scores_b)   # Welch の t 検定（既定）

In [ ]:
# 分散が等しいと仮定できる場合は var.equal = TRUE（スチューデントの t 検定）
t.test(scores_a, scores_b, var.equal = TRUE)

### 2.3 対応のある t 検定

**同じ人** の研修前後の点数のように、ペアになったデータには `paired = TRUE` を使います。
差の平均が 0 かどうかを検定していることと同じです。

In [ ]:
set.seed(2)
before <- round(rnorm(20, mean = 60, sd = 8))            # 研修前
after  <- before + round(rnorm(20, mean = 4, sd = 5))    # 研修後（平均 4 点アップ）

print(head(data.frame(before, after, diff = after - before)))
t.test(after, before, paired = TRUE)

### 片側検定

「研修**後のほうが高い**」ことだけを検定したいときは `alternative = "greater"` を指定します
（既定は両側 `"two.sided"`）。

In [ ]:
t.test(after, before, paired = TRUE, alternative = "greater")

### 練習問題 2

ある工場の部品の重さは仕様では 250 g です。
`set.seed(7); weight <- rnorm(30, mean = 252, sd = 5)` でデータを作り、
1. 1 標本 t 検定で「平均は 250 g と言えるか」を検定してください
2. p 値と 95% 信頼区間から結論を述べてください（コメントで OK）
3. 「250 g より**重い**」という片側検定も行ってください

In [ ]:
# 練習問題 2 の解答欄：ここにコードを書いてください

<details>
<summary><strong>練習問題 2 の解答例を見る</strong></summary>

```r
set.seed(7)
weight <- rnorm(30, mean = 252, sd = 5)

print(t.test(weight, mu = 250))
# p 値が 0.05 を下回れば「平均 250 g とは言えない」。
# 信頼区間に 250 が含まれないことも同じ結論を支持する。

t.test(weight, mu = 250, alternative = "greater")
```

</details>

---
## 3. 分散の検定（F 検定）

2 群の**ばらつき（分散）**が等しいかは `var.test()` で検定できます（帰無仮説: 分散比 = 1）。
2 標本 t 検定で `var.equal = TRUE` を使ってよいかの目安になります。
ただし F 検定は正規性からのずれに敏感なので、迷ったら Welch の t 検定をそのまま使うのが実務的です。

In [ ]:
var.test(scores_a, scores_b)

---
## 4. 比率の検定

「購入率」「クリック率」のような **割合** に関する検定です。

| 関数 | 特徴 |
|---|---|
| `binom.test()` | 二項分布そのものを使う正確検定（1 標本） |
| `prop.test()` | 正規近似。1 標本にも 2 標本（比率の差）にも使える |

### 4.1 1 標本：購入率は 15% と言えるか

200 人中 45 人が購入したとします（標本比率 22.5%）。

In [ ]:
prop.test(x = 45, n = 200, p = 0.15)

In [ ]:
binom.test(45, 200, p = 0.15)   # 正確検定でも同じ結論になるか確認

### 4.2 2 標本：A/B テスト

Web ページのデザイン A（180 人中 48 人が登録）とデザイン B（170 人中 33 人が登録）で
登録率に差があるかを検定します。

In [ ]:
prop.test(x = c(48, 33), n = c(180, 170))

### 練習問題 3

ある広告のクリック率は従来 4% でした。新しい広告を 1500 回表示したところ 78 回クリックされました。
1. `prop.test()` で「クリック率は 4% から変わったか」を検定してください
2. 標本のクリック率（78/1500）も計算して表示してください

In [ ]:
# 練習問題 3 の解答欄：ここにコードを書いてください

<details>
<summary><strong>練習問題 3 の解答例を見る</strong></summary>

```r
cat("標本クリック率:", round(78 / 1500 * 100, 2), "%\n")
prop.test(x = 78, n = 1500, p = 0.04)
# p < 0.05 なら「4% から変わった」と判断（今回は上昇方向）
```

</details>

---
## 5. カイ二乗検定

**度数（カウント）データ** の検定です。`chisq.test()` を 2 通りに使います。

### 5.1 適合度検定

「観測された度数が、理論上の比率と合っているか」を調べます。
例：市場シェアが A 35%・B 30%・C 20%・D 15% と言われているブランドについて、
200 人に聞いた選好が理論比率と合っているか。

In [ ]:
obs <- c(A = 72, B = 55, C = 48, D = 25)   # 観測度数（合計 200）
print(obs)
chisq.test(obs, p = c(0.35, 0.30, 0.20, 0.15))

### 5.2 独立性の検定

クロス集計表（分割表）で「2 つの質的変数に関連があるか」を調べます。
例：性別と購入の有無は独立か。

In [ ]:
purchase <- matrix(c(35, 45,
                     65, 55),
                   nrow = 2, byrow = TRUE,
                   dimnames = list(sex = c("Male", "Female"),
                                   buy = c("Yes", "No")))
print(purchase)
res_chi <- chisq.test(purchase)
print(res_chi)
cat("期待度数:\n")
print(round(res_chi$expected, 1))

- `X-squared`: カイ二乗統計量。観測度数と期待度数のずれの大きさ
- 2×2 表では既定で **イェーツの連続性補正** が入ります（`correct = FALSE` で無効化）
- **期待度数が 5 未満のセルがある** ときはカイ二乗近似が悪くなり警告が出ます。
  その場合は正確検定 `fisher.test()` を使います

In [ ]:
# 度数が小さい表はフィッシャーの正確検定を使う
small <- matrix(c(3, 9,
                  8, 2), nrow = 2, byrow = TRUE,
                dimnames = list(group = c("G1", "G2"), result = c("Hit", "Miss")))
print(small)
fisher.test(small)

### 練習問題 4

サイコロを 120 回振ったところ、1〜6 の目がそれぞれ `c(15, 18, 25, 20, 24, 18)` 回出ました。
1. このサイコロは公正（各目の確率 1/6）と言えるか、適合度検定で調べてください
2. 期待度数（各目 20 回）とのずれをコメントで説明してください

In [ ]:
# 練習問題 4 の解答欄：ここにコードを書いてください

<details>
<summary><strong>練習問題 4 の解答例を見る</strong></summary>

```r
dice <- c(15, 18, 25, 20, 24, 18)
chisq.test(dice, p = rep(1/6, 6))
# p 値が大きい → 「公正でない」とは言えない（観測度数のばらつきは偶然の範囲）
```

</details>

---
## 6. 一元配置分散分析（ANOVA）

**3 群以上** の平均を比べるときは、t 検定を繰り返すのではなく分散分析を使います
（繰り返すと偶然の有意差が出やすくなるため。9 章参照）。

例：3 つの地域（East / Central / West)の店舗売上に差があるか。
- 帰無仮説 H0: 3 地域の母平均はすべて等しい

In [ ]:
set.seed(3)
region <- factor(rep(c("East", "Central", "West"), each = 25))
sales  <- round(c(rnorm(25, 100, 12), rnorm(25, 108, 12), rnorm(25, 96, 12)))

cat("地域ごとの平均:\n")
print(round(tapply(sales, region, mean), 1))
boxplot(sales ~ region, main = "Sales by region",
        xlab = "Region", ylab = "Sales", col = "lightgreen")

In [ ]:
res_aov <- aov(sales ~ region)
summary(res_aov)

- `F value`: 群間のばらつき ÷ 群内のばらつき。大きいほど「群間に差がある」
- `Pr(>F)` が 0.05 未満なら「少なくとも 1 組の平均に差がある」と判断します

### どの組に差があるか：Tukey の多重比較

ANOVA は「どこかに差がある」ことしか教えてくれません。
どのペアに差があるかは `TukeyHSD()` で調べます（多重比較の補正込み）。

In [ ]:
TukeyHSD(res_aov)

In [ ]:
plot(TukeyHSD(res_aov))   # 信頼区間が 0 をまたがないペアに有意差がある

分散が等しい仮定を置きたくないときは、Welch 版の一元配置検定 `oneway.test()` が使えます。

In [ ]:
oneway.test(sales ~ region)

### 練習問題 5

3 種類の肥料 F1・F2・F3 で育てた植物の高さを
`set.seed(9); height <- c(rnorm(20, 30, 4), rnorm(20, 34, 4), rnorm(20, 31, 4))`、
`fert <- factor(rep(c("F1", "F2", "F3"), each = 20))` で作り、
1. 肥料ごとの平均を表示し、箱ひげ図を描いてください
2. 分散分析で肥料の効果を検定してください
3. `TukeyHSD()` でどのペアに差があるか確認してください

In [ ]:
# 練習問題 5 の解答欄：ここにコードを書いてください

<details>
<summary><strong>練習問題 5 の解答例を見る</strong></summary>

```r
set.seed(9)
height <- c(rnorm(20, 30, 4), rnorm(20, 34, 4), rnorm(20, 31, 4))
fert <- factor(rep(c("F1", "F2", "F3"), each = 20))

print(round(tapply(height, fert, mean), 1))
boxplot(height ~ fert, main = "Height by fertilizer", xlab = "Fertilizer", ylab = "Height", col = "wheat")

res <- aov(height ~ fert)
print(summary(res))
TukeyHSD(res)   # F2-F1 の差が最も大きいはず
```

</details>

---
## 7. ノンパラメトリック検定

データが正規分布から大きく外れているときや、5 段階評価のような**順序データ**のときは、
分布の形を仮定しないノンパラメトリック検定を使います。

| パラメトリック | ノンパラメトリック対応 |
|---|---|
| 2 標本 t 検定 | `wilcox.test(x, y)`（マン・ホイットニーの U 検定） |
| 対応あり t 検定 | `wilcox.test(x, y, paired = TRUE)`（符号付き順位検定） |
| 一元配置 ANOVA | `kruskal.test()`（クラスカル・ウォリス検定） |

5 段階満足度のように**同順位（タイ）が多いデータ**では、正確な p 値が計算できないという警告が出るため、
`exact = FALSE`（正規近似）を指定します。

In [ ]:
set.seed(4)
satis_x <- sample(1:5, 30, replace = TRUE, prob = c(0.10, 0.15, 0.30, 0.30, 0.15))  # 店舗 X の満足度
satis_y <- sample(1:5, 28, replace = TRUE, prob = c(0.05, 0.10, 0.20, 0.35, 0.30))  # 店舗 Y の満足度

cat("店舗 X:", "\n"); print(table(satis_x))
cat("店舗 Y:", "\n"); print(table(satis_y))
wilcox.test(satis_x, satis_y, exact = FALSE)

In [ ]:
# 対応ありの場合（2 章の研修前後データを再利用）
wilcox.test(after, before, paired = TRUE, exact = FALSE)

In [ ]:
# 3 群以上（6 章の地域別売上データを再利用）
kruskal.test(sales ~ region)

### 練習問題 6

2 つのサポート窓口の応対評価（1〜5）を
`set.seed(11); desk_a <- sample(1:5, 25, replace = TRUE, prob = c(.2, .3, .3, .15, .05))`、
`desk_b <- sample(1:5, 25, replace = TRUE, prob = c(.05, .15, .3, .3, .2))` で作り、
マン・ホイットニーの U 検定で評価に差があるか調べてください（`exact = FALSE` を忘れずに）。

In [ ]:
# 練習問題 6 の解答欄：ここにコードを書いてください

<details>
<summary><strong>練習問題 6 の解答例を見る</strong></summary>

```r
set.seed(11)
desk_a <- sample(1:5, 25, replace = TRUE, prob = c(.2, .3, .3, .15, .05))
desk_b <- sample(1:5, 25, replace = TRUE, prob = c(.05, .15, .3, .3, .2))

print(table(desk_a))
print(table(desk_b))
wilcox.test(desk_a, desk_b, exact = FALSE)   # p < 0.05 なら評価の分布に差がある
```

</details>

---
## 8. 相関の検定

2 つの量的変数の関係の強さは相関係数で測り、`cor.test()` で
「母相関 = 0（無相関）」を帰無仮説として検定できます。

- `method = "pearson"`（既定）: 直線的な関係。正規性を仮定
- `method = "spearman"`: 順位に基づく。外れ値や非線形な単調関係に頑健

In [ ]:
set.seed(5)
adspend <- runif(40, 10, 100)                       # 広告費（万円）
revenue <- 50 + 1.8 * adspend + rnorm(40, 0, 25)    # 売上（万円）

plot(adspend, revenue, main = "Ad spend vs revenue",
     xlab = "Ad spend", ylab = "Revenue", pch = 19, col = "steelblue")
cat("Pearson の相関係数:", round(cor(adspend, revenue), 3), "\n")

In [ ]:
cor.test(adspend, revenue)

In [ ]:
cor.test(adspend, revenue, method = "spearman")

> **注意**: 相関があっても因果があるとは限りません（疑似相関・交絡）。
> 「アイスの売上と水難事故」はどちらも気温の影響で相関しますが、因果関係はありません。

---
## 9. p 値と効果量の考え方

### p 値の落とし穴

p 値は「差の大きさ」ではありません。**標本サイズが大きいと、実務的には無意味な差でも p < 0.05 になります。**

In [ ]:
# 効果量 Cohen's d（2 群の平均差を標準偏差で割ったもの）を計算する関数
cohens_d <- function(x, y) {
  nx <- length(x); ny <- length(y)
  sp <- sqrt(((nx - 1) * var(x) + (ny - 1) * var(y)) / (nx + ny - 2))  # プールした標準偏差
  (mean(x) - mean(y)) / sp
}

cat("クラス A vs B の Cohen's d:", round(cohens_d(scores_a, scores_b), 3), "\n")
# 目安: 0.2 = 小さい / 0.5 = 中くらい / 0.8 = 大きい

In [ ]:
# 10 万人ずつのデータでは、平均差わずか 0.2 でも「有意」になる
set.seed(6)
big_x <- rnorm(100000, mean = 100.0, sd = 15)
big_y <- rnorm(100000, mean = 100.2, sd = 15)

res_big <- t.test(big_x, big_y)
cat("p 値       :", signif(res_big$p.value, 3), "\n")
cat("平均差     :", round(unname(res_big$estimate[1] - res_big$estimate[2]), 3), "\n")
cat("Cohen's d  :", round(cohens_d(big_x, big_y), 4), "  ← ごく小さい効果\n")

**p 値と効果量はセットで報告する**のが現代の標準です。

### 多重比較の問題

検定を繰り返すほど「偶然の有意差」が出やすくなります（20 回繰り返せば約 64% の確率で少なくとも 1 回は p < 0.05 が出ます）。
複数の p 値をまとめて補正するには `p.adjust()` を使います。

In [ ]:
pvals <- c(0.010, 0.020, 0.030, 0.040, 0.200)   # 5 回の検定で得た p 値

print(p.adjust(pvals, method = "bonferroni"))   # ボンフェローニ（最も保守的）
print(p.adjust(pvals, method = "holm"))         # ホルム（既定。ボンフェローニより効率的）
print(p.adjust(pvals, method = "BH"))           # Benjamini-Hochberg（偽発見率の制御）

### 練習問題 7

`set.seed(20)` のもとで、**まったく同じ分布** `rnorm(30, 50, 10)` から 2 群を取り出して
t 検定する、という操作を 100 回繰り返し、p < 0.05 になった回数を数えてください
（帰無仮説が正しいのに「有意」になる確率が約 5% あることを確かめる実験です）。

In [ ]:
# 練習問題 7 の解答欄：ここにコードを書いてください

<details>
<summary><strong>練習問題 7 の解答例を見る</strong></summary>

```r
set.seed(20)
n_sig <- 0
for (i in 1:100) {
  g1 <- rnorm(30, 50, 10)
  g2 <- rnorm(30, 50, 10)
  if (t.test(g1, g2)$p.value < 0.05) n_sig <- n_sig + 1
}
cat("100 回中 p < 0.05 になった回数:", n_sig, "\n")
# おおよそ 5 回前後になる（これが「第一種の過誤」の確率 5%）
```

</details>

---
## 10. まとめ

| 目的 | データ | 関数 |
|---|---|---|
| 平均 vs 基準値 | 量的・1 群 | `t.test(x, mu =)` |
| 2 群の平均差 | 量的・独立 2 群 | `t.test(x, y)`（Welch） |
| 前後比較 | 量的・対応あり | `t.test(x, y, paired = TRUE)` |
| 分散の比較 | 量的・2 群 | `var.test(x, y)` |
| 比率 | 割合 | `prop.test()` / `binom.test()` |
| 度数と理論比率 | カウント | `chisq.test(x, p =)` |
| クロス表の関連 | カウント | `chisq.test(表)` / `fisher.test(表)` |
| 3 群以上の平均 | 量的・多群 | `aov()` → `TukeyHSD()` |
| 順序・非正規データ | 順位 | `wilcox.test()` / `kruskal.test()` |
| 相関 | 量的・2 変数 | `cor.test()` |

**検定の流れ**: 分布を図で確認 → 適切な検定を選ぶ → p 値と**効果量・信頼区間**をセットで解釈

## 次のステップ
- [R 回帰分析入門](r_regression_beginner_tutorial.ipynb) — 検定の次は、変数の関係をモデル化する回帰分析へ
- [R 文法入門](r_beginner_tutorial.ipynb) — R の文法に不安があればこちらから

### 総合演習

あるオンラインショップで、新しいレコメンド機能の効果を検証します。次のデータを作って分析してください。

```r
set.seed(50)
old_amount <- rnorm(80, mean = 3200, sd = 900)   # 旧機能ユーザーの購入額
new_amount <- rnorm(80, mean = 3600, sd = 950)   # 新機能ユーザーの購入額
old_click <- 96    # 旧: 800 表示中 96 クリック
new_click <- 132   # 新: 800 表示中 132 クリック
```

1. 購入額の分布をヒストグラムで比べ、2 標本 t 検定で平均差を検定する
2. 効果量 Cohen's d を計算する（9 章の関数を再利用）
3. クリック率の差を `prop.test()` で検定する
4. 結果を 2〜3 行のコメントでまとめる

In [ ]:
# 総合演習 の解答欄：ここにコードを書いてください

<details>
<summary><strong>総合演習 の解答例を見る</strong></summary>

```r
set.seed(50)
old_amount <- rnorm(80, mean = 3200, sd = 900)
new_amount <- rnorm(80, mean = 3600, sd = 950)

# 1. 分布の比較と t 検定
par(mfrow = c(1, 2))
hist(old_amount, main = "Old", xlab = "Amount", col = "gray")
hist(new_amount, main = "New", xlab = "Amount", col = "skyblue")
par(mfrow = c(1, 1))
print(t.test(old_amount, new_amount))

# 2. 効果量
cat("Cohen's d:", round(cohens_d(new_amount, old_amount), 3), "\n")

# 3. クリック率の検定
print(prop.test(x = c(132, 96), n = c(800, 800)))

# 4. まとめ（例）:
# 購入額は新機能のほうが平均で約 400 高く、t 検定で有意（効果量は中程度）。
# クリック率も 12.0% → 16.5% に上昇し、比率の検定でも有意。
# 新レコメンド機能には効果があると判断できる。
```

</details>